In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

# BraTS SSL split

Train / val split for SSL pretraining checkpoint selection.

In [2]:
import json, os, random

tensor_files = sorted(f for f in os.listdir(CONFIG.brats_tensors_96) if f.endswith('.pt'))
all_ids = [f.removesuffix('.pt') for f in tensor_files]

if os.path.exists(CONFIG.brats_overlap_ids_path):
    with open(CONFIG.brats_overlap_ids_path) as f:
        overlap_ids = set(json.load(f) or [])
else:
    overlap_ids = set()

clean_ids = sorted(set(all_ids) - overlap_ids)

random.seed(CONFIG.seed)
random.shuffle(clean_ids)
val_size = max(1, int(len(clean_ids) * CONFIG.test_split))

for path, ids in [(CONFIG.brats_ssl_train_ids_path, clean_ids[val_size:]),
                   (CONFIG.brats_ssl_val_ids_path,   clean_ids[:val_size])]:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        json.dump(ids, f)

print(f'{len(all_ids):>5} tensors')
print(f'{len(overlap_ids):>5} overlap (excluded)')
print(f'{len(clean_ids) - val_size:>5} train  -> {os.path.basename(CONFIG.brats_ssl_train_ids_path)}')
print(f'{val_size:>5} val    -> {os.path.basename(CONFIG.brats_ssl_val_ids_path)}')

 1251 tensors
    0 overlap (excluded)
 1126 train  -> ssl_train_ids.json
  125 val    -> ssl_val_ids.json
